In [1]:
import sys
import os
import pandas as pd
import numpy as np
import yaml
from DaySim import DaysimSummary
from Survey import DaysimSummary_Survey
from calibration_utils import (load_beta_lookup, save_log, write_f12, run_1d, run_2d, _make_csv_key_fn, _get_raw)

## Run once per session

In [2]:
# Load model and survey objects
model  = DaysimSummary()
survey = DaysimSummary_Survey()

runVehAvailability = True, loading data...
runWrkSchLocationChoice = True, loading data...
runTripMode = True, loading data...
runTourMode = True, loading data...
runTripDestination = True, loading data...
runTourDestination = True, loading data...
runTripTOD = True, loading data...
runTourTOD = True, loading data...
runDayPattern = True, loading data...
runVehAvailability = True, loading data...
runWrkSchLocationChoice = True, loading data...
runTripMode = True, loading data...
runTourMode = True, loading data...
runTripDestination = True, loading data...
runTourDestination = True, loading data...
runTripTOD = True, loading data...
runTourTOD = True, loading data...
runDayPattern = True, loading data...


## Run for each calibration run
      - select model to run in calibration_config.yaml and run cells below
      - initially, review LOG_FILE and updated coefficient file in coefficient_files_interim_dir after each run. If works as expected, manually copy to the coefficient_files_input_dir to prepare for the next run
      - option to enable COPY_BACK to overwrite input coefficient file with updated coefficient file at the end of each run instead of manually copying

In [3]:
# Load config
with open('calibration_config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

DAMPING_FACTOR = cfg['damping_factor']
THRESHOLD = cfg['threshold']   
COPY_BACK = cfg['copy_back']

TARGETS_FILE = cfg['paths']['targets_file']
MAPPING_FILE = cfg['paths']['mapping_file']  
LOG_FILE = cfg['paths']['log_file']

coefficient_files_input_dir = cfg['paths']['coefficient_files_input_dir']
coefficient_files_interim_dir = cfg['paths']['coefficient_files_interim_dir']

MODELS_TO_RUN = [m for m, enabled in cfg['models'].items() if enabled]

print(f'Models running calibration: {MODELS_TO_RUN}')

# Load targets and mapping 
targets_df = pd.read_csv(TARGETS_FILE)
targets_df["method_arg"] = targets_df["method_arg"].fillna('')
targets_df['key_mode'] = targets_df['key_mode'].fillna('')

mapping_df = pd.read_csv(MAPPING_FILE)
mapping_df["group"] = mapping_df["group"].fillna('').astype(str)
mapping_df["alternative"] = mapping_df["alternative"].astype(str)
mapping_df["end"] = pd.to_numeric(mapping_df["end"], errors='coerce')


Models running calibration: ['WorkLocation', 'SchoolLocation', 'PayToParkAtWorkplace', 'AutoOwnership', 'IndividualPersonDayPattern', 'PersonExactNumberOfTours', 'WorkTourMode', 'EscortTourMode', 'OtherHomeBasedTourMode', 'WorkBasedSubtourMode', 'WorkTourTime', 'SchoolTourTime']


In [4]:

# Coefficient files
F12_FILES = {
    file.split('Coefficients_Chattanooga')[0]: file 
    for file in os.listdir(coefficient_files_input_dir) 
    if file.endswith('.F12')
}

# Run calibration for each enabled model
for MODEL in MODELS_TO_RUN:
    print(f'Running calibration for model {MODEL}')
    model_targets = targets_df[targets_df['model'] == MODEL]
    if model_targets.empty:
        print(f'No targets found for model {MODEL}. Exiting.')
        exit()

    f12_models_needed = model_targets['f12_model'].unique().tolist()
    mapping_sub = mapping_df[mapping_df['f12_model'].isin(f12_models_needed)]
    beta_lookup = load_beta_lookup(mapping_sub, F12_FILES, coefficient_files_input_dir)
    #print(list(beta_lookup.keys())[:5])  # Print first 5 keys to verify loading
# Run calibration
    log = []
    for _, cfg in model_targets.iterrows():
        try:
            m_raw = _get_raw(model, cfg['method'], cfg['method_arg'])
            s_raw = _get_raw(survey, cfg['method'], cfg['method_arg'])
            csv_key_fn = _make_csv_key_fn(cfg['key_mode'], cfg['method_arg'])
            runner = run_2d if cfg['type'] == '2d' else run_1d
            log += runner(cfg['label'], cfg['f12_model'], m_raw, s_raw, 
                               beta_lookup, DAMPING_FACTOR, THRESHOLD)
        except Exception as e:
            print(f'Error processing target {cfg["label"]}: {e}')

    comp_df = pd.DataFrame(log)
    calibration_rows= comp_df[comp_df['calibrate']]
    n_within = calibration_rows['within_threshold'].sum()
    n_total = len(calibration_rows)
    print(f'Calibration {MODEL} complete: {n_within} of {n_total} already within threshold; '
          f'{n_total - n_within} betas updated for next run.')


    # Save log
    save_log(comp_df, LOG_FILE)

    # Write updated F12 files
    to_update = comp_df[comp_df['calibrate'] & 
                        ~comp_df['within_threshold'] & 
                        (comp_df['adjustment']!= 0) &
                        comp_df['end'].notna() &
                        comp_df['new_beta'].notna()]

    if to_update.empty:
        print('No updates needed for F12 files based on calibration results.')
    else:
        print(f'Updating F12 files for {to_update["f12model"].nunique()} models based on calibration results...')
        for f12_model,grp in to_update.groupby('f12model'):
            if f12_model not in F12_FILES:
                print(f'Warning: No F12 file found for model {f12_model}, skipping update.')
                continue
            end_to_beta = {int(r['end']): round(r['new_beta'], 12) 
                        for _, r in grp.iterrows()}
            write_f12(f12_model, 
                    end_to_beta, 
                    f12_files = F12_FILES, 
                    coefficient_input_dir = coefficient_files_input_dir,
                    coefficient_interim_dir = coefficient_files_interim_dir,
                    copy_back = COPY_BACK)


Running calibration for model WorkLocation
('run_id', '0-3.5 mi', '', '0-3.5 mi')
('WorkLocation', '', '0-3.5 mi')
('run_id', '3.5-10 mi', '', '3.5-10 mi')
('WorkLocation', '', '3.5-10 mi')
('run_id', '10+ mi', '', '10+ mi')
('WorkLocation', '', '10+ mi')
Calibration WorkLocation complete: 0 of 3 already within threshold; 3 betas updated for next run.
Saved calibration_log.csv (run13)
Updating F12 files for 1 models based on calibration results...
  WorkLocationCoefficients_Chattanooga.F12: 3 Beta(s) updated  ->  data/interim\WorkLocationCoefficients_Chattanooga_20.F12
 COPY_BACK is False: Original file 9_Coefficients\WorkLocationCoefficients_Chattanooga.F12 remains unchanged
Running calibration for model SchoolLocation
('SchoolLocation', 'Ch515', '0-1 mi')
('SchoolLocation', 'Ch515', '1-5 mi')
('SchoolLocation', 'Ch515', '5+ mi')
('SchoolLocation', 'Stu16', '0-1 mi')
('SchoolLocation', 'Stu16', '1-5 mi')
('SchoolLocation', 'Stu16', '5+ mi')
('SchoolLocation', 'UniStu', '0-1 mi')
('Sch